# Multithreading & Multiprocessing — Assignment 2

Thirty solved problems progressing from **medium → challenging**. Thread examples execute in this notebook. Multiprocessing launch examples are defined but not started inside Jupyter because Windows/spawn requires importable top-level functions in a `.py` file; each such question clearly shows the expected output when saved and run.

In [1]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import multiprocessing as mp
from multiprocessing import shared_memory
import os
import queue
import threading
import time

## Question 1 — Medium

**Concept tested:** Workload selection

**Question:** Write a function that recommends threads for I/O-bound work and processes for CPU-bound work.

**Hint:** Normalize the input string.

### Solution approach

Map two known workload types and reject unknown values.

In [2]:
def choose_worker(workload):
    normalized = workload.strip().lower()
    if normalized == "io":
        return "threads"
    if normalized == "cpu":
        return "processes"
    raise ValueError("workload must be 'io' or 'cpu'")
print(choose_worker("IO"), choose_worker("cpu"))

threads processes


### Expected output

```text
threads processes
```

### Step-by-step explanation

Waiting tasks benefit from thread concurrency; pure-Python computation can use separate interpreters/cores through processes.

### What was learned

Classify and measure workloads before choosing a concurrency tool.

## Question 2 — Medium

**Concept tested:** Thread start/join

**Question:** Run two short workers and wait for both.

**Hint:** Call `start()` before `join()`.

### Solution approach

Create named threads that append completion records.

In [3]:
finished = []
def worker(name):
    time.sleep(0.01)
    finished.append(name)
threads = [threading.Thread(target=worker, args=(name,)) for name in ["A", "B"]]
for thread in threads: thread.start()
for thread in threads: thread.join()
print(sorted(finished))

['A', 'B']


### Expected output

```text
['A', 'B']
```

### Step-by-step explanation

`start` schedules workers; `join` creates a clear completion boundary before reading results.

### What was learned

Thread output order is nondeterministic; coordinate before consuming shared results.

## Question 3 — Medium

**Concept tested:** Returning thread results

**Question:** Collect a thread result safely using `queue.Queue`.

**Hint:** Put in the worker; get in the parent.

### Solution approach

Use a thread-safe message queue rather than a global result race.

In [4]:
results = queue.Queue()
def square_worker(value):
    results.put(value * value)
t = threading.Thread(target=square_worker, args=(7,))
t.start(); t.join()
print(results.get())

49


### Expected output

```text
49
```

### Step-by-step explanation

Queue synchronizes producer and consumer and transfers ownership of the message.

### What was learned

Plain Thread has no return value; Queue or Future carries results.

## Question 4 — Medium

**Concept tested:** ThreadPoolExecutor.map

**Question:** Square four values concurrently while keeping input order.

**Hint:** `map` returns results in input order.

### Solution approach

Use a two-worker pool and convert results to a list.

In [5]:
def slow_square(value):
    time.sleep(0.005)
    return value * value
with ThreadPoolExecutor(max_workers=2) as pool:
    values = list(pool.map(slow_square, [1, 2, 3, 4]))
print(values)

[1, 4, 9, 16]


### Expected output

```text
[1, 4, 9, 16]
```

### Step-by-step explanation

The pool reuses workers; `map` preserves ordering even if completion differs.

### What was learned

Pools simplify lifecycle and bound concurrency.

## Question 5 — Medium

**Concept tested:** Future errors

**Question:** Submit one successful and one failing task, then handle each Future.

**Hint:** Exceptions reappear at `future.result()`.

### Solution approach

Map futures to inputs and catch `ZeroDivisionError`.

In [6]:
def reciprocal(value):
    return 1 / value
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {pool.submit(reciprocal, x): x for x in [2, 0]}
    outcomes = []
    for future, value in futures.items():
        try: outcomes.append((value, future.result()))
        except ZeroDivisionError: outcomes.append((value, "error"))
print(sorted(outcomes, key=lambda x: x[0]))

[(0, 'error'), (2, 0.5)]


### Expected output

```text
[(0, 'error'), (2, 0.5)]
```

### Step-by-step explanation

Worker exceptions are stored by Future and raised when the result is requested.

### What was learned

Always observe Futures or task failures may be missed.

## Question 6 — Medium

**Concept tested:** Race conditions and Lock

**Question:** Increment a shared counter from four threads safely.

**Hint:** Protect the read-change-write section.

### Solution approach

Use one lock around every increment.

In [7]:
counter = 0
lock = threading.Lock()
def increment(times):
    global counter
    for _ in range(times):
        with lock:
            counter += 1
threads = [threading.Thread(target=increment, args=(1000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(counter)

4000


### Expected output

```text
4000
```

### Step-by-step explanation

The lock makes each compound update exclusive. Never rely on the GIL as a data-integrity lock.

### What was learned

Protect shared invariants, not just individual lines.

## Question 7 — Medium

**Concept tested:** RLock

**Question:** Write nested methods that acquire the same lock without self-deadlocking.

**Hint:** Use `threading.RLock`.

### Solution approach

Have an outer method call an inner locked method.

In [8]:
class Counter:
    def __init__(self):
        self.value = 0
        self.lock = threading.RLock()
    def add_one(self):
        with self.lock: self.value += 1
    def add_two(self):
        with self.lock:
            self.add_one(); self.add_one()
c = Counter(); c.add_two(); print(c.value)

2


### Expected output

```text
2
```

### Step-by-step explanation

RLock tracks owning thread and recursion depth, allowing the same thread to reacquire it.

### What was learned

Use RLock for genuinely nested locking; prefer simpler Lock otherwise.

## Question 8 — Medium

**Concept tested:** Semaphore

**Question:** Limit a section to at most two simultaneous threads and verify the peak.

**Hint:** Track active count under a lock.

### Solution approach

Wrap work in `Semaphore(2)` and record the maximum.

In [9]:
gate = threading.Semaphore(2)
state_lock = threading.Lock()
active = peak = 0
def limited():
    global active, peak
    with gate:
        with state_lock:
            active += 1; peak = max(peak, active)
        time.sleep(0.01)
        with state_lock: active -= 1
threads = [threading.Thread(target=limited) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(peak)

2


### Expected output

```text
2
```

### Step-by-step explanation

The semaphore owns two permits; other threads wait until a permit is released.

### What was learned

Semaphores protect limited-capacity resources such as connection pools.

## Question 9 — Medium

**Concept tested:** Event

**Question:** Make workers wait until configuration is ready.

**Hint:** Use `Event.wait()` and `set()`.

### Solution approach

Start waiters, set shared data, then signal them.

In [10]:
ready = threading.Event()
config = {}
seen = []
def reader():
    ready.wait()
    seen.append(config["mode"])
t = threading.Thread(target=reader); t.start()
config["mode"] = "safe"
ready.set(); t.join()
print(seen)

['safe']


### Expected output

```text
['safe']
```

### Step-by-step explanation

The event establishes a simple signal after configuration is populated.

### What was learned

Events express readiness/stop signals without polling.

## Question 10 — Medium

**Concept tested:** Producer-consumer Queue

**Question:** Send three jobs and a sentinel from producer to consumer.

**Hint:** Call `task_done()` for every `get()`.

### Solution approach

Use `None` as a stop message and wait with `join()`.

In [11]:
jobs = queue.Queue(); processed = []
def consumer():
    while True:
        item = jobs.get()
        try:
            if item is None: return
            processed.append(item * 2)
        finally: jobs.task_done()
t = threading.Thread(target=consumer); t.start()
for item in [1, 2, 3]: jobs.put(item)
jobs.put(None); jobs.join(); t.join()
print(processed)

[2, 4, 6]


### Expected output

```text
[2, 4, 6]
```

### Step-by-step explanation

Queue blocks safely, sentinel ends the loop, and task accounting lets the producer wait for completion.

### What was learned

Queues reduce direct shared-state coordination.

## Question 11 — Medium

**Concept tested:** Barrier

**Question:** Make three threads wait at a checkpoint before recording completion.

**Hint:** Use `threading.Barrier(3)`.

### Solution approach

Each worker appends before, waits, then appends after.

In [12]:
barrier = threading.Barrier(3)
events = []
events_lock = threading.Lock()
def checkpoint(n):
    with events_lock: events.append(f"before{n}")
    barrier.wait()
    with events_lock: events.append(f"after{n}")
threads = [threading.Thread(target=checkpoint, args=(n,)) for n in range(3)]
for t in threads: t.start()
for t in threads: t.join()
first_after = min(i for i, value in enumerate(events) if value.startswith("after"))
print(first_after == 3)

True


### Expected output

```text
True
```

### Step-by-step explanation

No thread can pass until all three have reached the barrier.

### What was learned

Barriers coordinate phases with a fixed participant count.

## Question 12 — Medium

**Concept tested:** Deadlock prevention

**Question:** Acquire two locks in one consistent global order.

**Hint:** Sort resources by a stable key.

### Solution approach

Create a helper that orders locks by `id` before entering.

In [13]:
lock_a, lock_b = threading.Lock(), threading.Lock()
def safe_two(first, second):
    ordered = sorted([first, second], key=id)
    with ordered[0]:
        with ordered[1]:
            return "done"
print(safe_two(lock_b, lock_a))

done


### Expected output

```text
done
```

### Step-by-step explanation

Every caller uses the same acquisition order, removing the circular-wait condition.

### What was learned

Consistent lock ordering is a primary deadlock-prevention technique.

## Question 13 — Medium

**Concept tested:** Timeout-aware join

**Question:** Detect that a worker has not completed without hanging forever.

**Hint:** Call `join(timeout=...)`, then `is_alive()`.

### Solution approach

Start a short sleeper and join for less than its duration.

In [14]:
def sleeper(): time.sleep(0.05)
t = threading.Thread(target=sleeper)
t.start(); t.join(timeout=0.005)
print("still working:", t.is_alive())
t.join()

still working: True


### Expected output

```text
still working: True
```

### Step-by-step explanation

Timed join returns control even when the worker continues.

### What was learned

Timeouts make waiting observable; threads cannot be safely force-killed.

## Question 14 — Medium

**Concept tested:** Thread-local state

**Question:** Give each thread an independent `request_id` despite sharing one local object.

**Hint:** Use `threading.local()`.

### Solution approach

Set/read an attribute inside each worker and collect results.

In [15]:
local = threading.local(); values = []; values_lock = threading.Lock()
def request_worker(request_id):
    local.request_id = request_id
    with values_lock: values.append(local.request_id)
threads = [threading.Thread(target=request_worker, args=(x,)) for x in ["A", "B"]]
for t in threads: t.start()
for t in threads: t.join()
print(sorted(values))

['A', 'B']


### Expected output

```text
['A', 'B']
```

### Step-by-step explanation

Each thread sees its own attribute storage on the same thread-local container.

### What was learned

Thread-local data avoids some sharing, but long-lived thread pools may retain it.

## Question 15 — Medium

**Concept tested:** GIL reasoning

**Question:** Explain in code which workload should use which executor.

**Hint:** Return classes, do not benchmark tiny work.

### Solution approach

Map labels to executor types.

In [16]:
def executor_for(kind):
    return ThreadPoolExecutor if kind == "io" else ProcessPoolExecutor
print(executor_for("io").__name__)
print(executor_for("cpu").__name__)

ThreadPoolExecutor


ProcessPoolExecutor


### Expected output

```text
ThreadPoolExecutor
ProcessPoolExecutor
```

### Step-by-step explanation

Threads overlap waits; processes can execute Python bytecode on multiple cores. Native libraries may release the GIL, so measure.

### What was learned

The GIL affects CPU-bound Python threads, not the existence/usefulness of concurrency.

## Question 16 — Challenging

**Concept tested:** Safe Process template

**Question:** Define a Windows-safe process demo with a top-level worker and main guard.

**Hint:** Do not launch it from the notebook.

### Solution approach

Define importable functions; show the guard in the solution.

In [17]:
def process_square(value):
    return value * value

def process_template():
    process = mp.Process(target=print, args=(process_square(5),))
    process.start(); process.join()
    return process.exitcode

print("Save to .py and call under: if __name__ == '__main__':")

Save to .py and call under: if __name__ == '__main__':


### Expected output

```text
Save to .py and call under: if __name__ == '__main__':
# Running process_template() there prints 25 and returns exit code 0.
```

### Step-by-step explanation

Spawn imports the module; the guard prevents each child from recursively creating children.

### What was learned

Top-level picklable workers and the main guard are required portable patterns.

## Question 17 — Challenging

**Concept tested:** Process timeout/exit code

**Question:** Define a helper that joins with a timeout and terminates a stuck process.

**Hint:** Check `is_alive()` and always join after terminate.

### Solution approach

Return a status string and inspect non-zero exits.

In [18]:
def supervise(process, timeout):
    process.start(); process.join(timeout)
    if process.is_alive():
        process.terminate(); process.join()
        return "timed out"
    return "ok" if process.exitcode == 0 else f"exit {process.exitcode}"
print("supervisor defined")

supervisor defined


### Expected output

```text
supervisor defined
```

### Step-by-step explanation

The parent regains control after the deadline and reaps the terminated child with a final join.

### What was learned

Processes can be terminated, but abrupt termination may skip cleanup or corrupt shared resources.

## Question 18 — Challenging

**Concept tested:** ProcessPoolExecutor

**Question:** Define a pool function that cubes values in parallel and preserves order.

**Hint:** Use top-level worker plus `executor.map`.

### Solution approach

Return a list from a guarded callable suitable for a script.

In [19]:
def process_cube(value):
    return value ** 3
def parallel_cubes(values):
    with ProcessPoolExecutor(max_workers=2) as pool:
        return list(pool.map(process_cube, values))
print([process_cube(x) for x in [1, 2, 3]])

[1, 8, 27]


### Expected output

```text
[1, 8, 27]
# parallel_cubes([1,2,3]) produces the same when run from a guarded .py file.
```

### Step-by-step explanation

Pool workers distribute independent inputs and map reconstructs input ordering.

### What was learned

Process pools are ideal for batches of independent CPU-heavy tasks.

## Question 19 — Challenging

**Concept tested:** Pool chunksize

**Question:** Explain and expose `chunksize` for many small process tasks.

**Hint:** Send groups to reduce IPC overhead.

### Solution approach

Define a guarded mapping function with a configurable chunksize.

In [20]:
def parallel_map_with_chunks(values, chunksize=100):
    with ProcessPoolExecutor() as pool:
        return list(pool.map(process_square, values, chunksize=chunksize))
print("chunksize must be measured for the real workload")

chunksize must be measured for the real workload


### Expected output

```text
chunksize must be measured for the real workload
```

### Step-by-step explanation

Larger chunks reduce scheduling/serialization messages but can hurt load balance when tasks vary.

### What was learned

Task granularity and serialization overhead strongly affect process performance.

## Question 20 — Challenging

**Concept tested:** Multiprocessing Queue

**Question:** Define parent/child message passing with a sentinel.

**Hint:** Close and join the queue feeder after use.

### Solution approach

Create a top-level producer and guarded runner.

In [21]:
def mp_producer(output):
    for value in [1, 2, 3]: output.put(value)
    output.put(None)
def queue_process_demo():
    output = mp.Queue(); process = mp.Process(target=mp_producer, args=(output,))
    process.start(); values = []
    while (item := output.get()) is not None: values.append(item)
    process.join(); output.close(); output.join_thread()
    return values
print("multiprocessing queue demo defined")

multiprocessing queue demo defined


### Expected output

```text
multiprocessing queue demo defined
# queue_process_demo() returns [1, 2, 3] in a guarded script.
```

### Step-by-step explanation

The queue serializes messages between isolated process memories; the sentinel ends consumption.

### What was learned

Close process queues and avoid waiting for a child before draining a full queue.

## Question 21 — Challenging

**Concept tested:** Pipe

**Question:** Define one child-to-parent message using a one-way pipe.

**Hint:** Close unused endpoints in both processes.

### Solution approach

Send a dictionary and close each connection after use.

In [22]:
def pipe_sender(connection):
    connection.send({"answer": 42}); connection.close()
def pipe_demo():
    receive_end, send_end = mp.Pipe(duplex=False)
    process = mp.Process(target=pipe_sender, args=(send_end,)); process.start()
    send_end.close(); message = receive_end.recv(); receive_end.close(); process.join()
    return message
print("one-way pipe demo defined")

one-way pipe demo defined


### Expected output

```text
one-way pipe demo defined
# pipe_demo() returns {'answer': 42} in a guarded script.
```

### Step-by-step explanation

A pipe has two endpoints; closing unused copies prevents confusing hangs/end-of-file behavior.

### What was learned

Pipes suit direct connections; queues suit broader producer-consumer patterns.

## Question 22 — Challenging

**Concept tested:** Shared Value + Lock

**Question:** Define a cross-process counter that reaches 10,000 safely.

**Hint:** A synchronized wrapper does not make a compound increment atomic; use a lock.

### Solution approach

Have two processes increment 5,000 times under one lock.

In [23]:
def mp_increment(counter, lock, times):
    for _ in range(times):
        with lock: counter.value += 1
def shared_counter_demo():
    counter, lock = mp.Value("i", 0), mp.Lock()
    workers = [mp.Process(target=mp_increment, args=(counter, lock, 5000)) for _ in range(2)]
    for worker in workers: worker.start()
    for worker in workers: worker.join()
    return counter.value
print("shared counter demo defined")

shared counter demo defined

### Expected output

```text
shared counter demo defined
# shared_counter_demo() returns 10000 in a guarded script.
```

### Step-by-step explanation

The lock protects the full read-modify-write operation across processes.

### What was learned

Shared memory needs explicit synchronization around invariants.

## Question 23 — Challenging

**Concept tested:** Shared Array

**Question:** Define workers that square different positions in a shared integer array.

**Hint:** Each process owns a distinct index.

### Solution approach

Pass index and Array to top-level workers.

In [24]:
def square_index(values, index):
    values[index] = values[index] ** 2
def shared_array_demo():
    values = mp.Array("i", [1, 2, 3])
    workers = [mp.Process(target=square_index, args=(values, i)) for i in range(3)]
    for worker in workers: worker.start()
    for worker in workers: worker.join()
    return list(values)
print("shared array demo defined")

shared array demo defined


### Expected output

```text
shared array demo defined
# shared_array_demo() returns [1, 4, 9] in a guarded script.
```

### Step-by-step explanation

Each worker writes a separate cell, avoiding a same-location race.

### What was learned

Partition ownership can reduce locking, but shared writes still require careful design.

## Question 24 — Challenging

**Concept tested:** Manager proxies

**Question:** Define a process-safe shared dictionary using `Manager`.

**Hint:** Manager operations use a server process and are slower than local objects.

### Solution approach

Create proxy, child writer, copy result before manager closes.

In [25]:
def manager_writer(shared):
    shared["status"] = "done"
def manager_demo():
    with mp.Manager() as manager:
        shared = manager.dict(); process = mp.Process(target=manager_writer, args=(shared,))
        process.start(); process.join(); return dict(shared)
print("manager demo defined")

manager demo defined


### Expected output

```text
manager demo defined
# manager_demo() returns {'status': 'done'} in a guarded script.
```

### Step-by-step explanation

The proxy forwards operations to a manager server, trading convenience for IPC overhead.

### What was learned

Prefer messages or simple shared values when a full proxy container is unnecessary.

## Question 25 — Challenging

**Concept tested:** Shared memory lifecycle

**Question:** Define a zero-copy byte-sharing example with correct close/unlink ownership.

**Hint:** Parent creates/unlinks; child attaches/closes.

### Solution approach

Pass the shared-memory name to a top-level child.

In [26]:
def edit_shared_byte(name):
    block = shared_memory.SharedMemory(name=name)
    try: block.buf[0] = 65
    finally: block.close()
def shared_memory_demo():
    block = shared_memory.SharedMemory(create=True, size=4)
    try:
        process = mp.Process(target=edit_shared_byte, args=(block.name,))
        process.start(); process.join(); return block.buf[0]
    finally:
        block.close(); block.unlink()
print("shared memory demo defined")

shared memory demo defined


### Expected output

```text
shared memory demo defined
# shared_memory_demo() returns 65 in a guarded script.
```

### Step-by-step explanation

Both processes attach to one buffer. Each closes its handle; the creator unlinks the named segment once.

### What was learned

Shared memory avoids copying but demands strict lifecycle and synchronization rules.

## Question 26 — Challenging

**Concept tested:** Sentinel pipeline

**Question:** Design a two-consumer thread pipeline that shuts down cleanly.

**Hint:** Send one sentinel per consumer.

### Solution approach

Workers double jobs, record under a lock, and acknowledge every item.

In [27]:
jobs = queue.Queue(); outputs = []; output_lock = threading.Lock()
def pipeline_worker():
    while True:
        item = jobs.get()
        try:
            if item is None: return
            with output_lock: outputs.append(item * 2)
        finally: jobs.task_done()
workers = [threading.Thread(target=pipeline_worker) for _ in range(2)]
for worker in workers: worker.start()
for item in range(5): jobs.put(item)
for _ in workers: jobs.put(None)
jobs.join()
for worker in workers: worker.join()
print(sorted(outputs))

[0, 2, 4, 6, 8]


### Expected output

```text
[0, 2, 4, 6, 8]
```

### Step-by-step explanation

One sentinel releases each consumer. Queue accounting waits until jobs and sentinels are handled.

### What was learned

Shutdown protocols are part of concurrent design, not an afterthought.

## Question 27 — Challenging

**Concept tested:** Cooperative cancellation

**Question:** Stop a long-running thread using an Event.

**Hint:** Threads should check a stop flag at safe points.

### Solution approach

Loop with `wait(timeout)` so the event both sleeps and wakes promptly.

In [28]:
stop = threading.Event(); ticks = []
def service():
    while not stop.wait(0.005): ticks.append("tick")
t = threading.Thread(target=service); t.start()
time.sleep(0.02); stop.set(); t.join()
print(len(ticks) > 0, not t.is_alive())

True True


### Expected output

```text
True True
```

### Step-by-step explanation

Setting the event wakes the wait and lets the worker return normally.

### What was learned

Python cannot safely kill arbitrary threads; design cooperative cancellation.

## Question 28 — Challenging

**Concept tested:** Hybrid architecture

**Question:** Choose a bounded thread stage for fetching and a process stage for CPU parsing.

**Hint:** Keep the code as a design function; avoid nesting uncontrolled pools.

### Solution approach

Return an architecture description with bounded worker counts.

In [29]:
def hybrid_plan(cpu_count):
    return {
        "fetch": {"executor": "ThreadPoolExecutor", "workers": 8},
        "compute": {"executor": "ProcessPoolExecutor", "workers": max(1, cpu_count - 1)},
        "handoff": "bounded queue",
    }
print(hybrid_plan(4))

{'fetch': {'executor': 'ThreadPoolExecutor', 'workers': 8}, 'compute': {'executor': 'ProcessPoolExecutor', 'workers': 3}, 'handoff': 'bounded queue'}


### Expected output

```text
{'fetch': {'executor': 'ThreadPoolExecutor', 'workers': 8}, 'compute': {'executor': 'ProcessPoolExecutor', 'workers': 3}, 'handoff': 'bounded queue'}
```

### Step-by-step explanation

Separate stages match tools to wait-heavy and compute-heavy work; a bounded handoff applies backpressure.

### What was learned

Hybrid systems need capacity limits and clear ownership more than extra workers.

## Question 29 — Challenging

**Concept tested:** Fair performance comparison

**Question:** Write a small timing helper and explain why one run is not proof.

**Hint:** Use `perf_counter`, repeats, and minimum/median.

### Solution approach

Measure a callable repeatedly and return the best time.

In [30]:
def best_time(function, repeats=5):
    samples = []
    for _ in range(repeats):
        start = time.perf_counter(); function(); samples.append(time.perf_counter() - start)
    return min(samples)
elapsed = best_time(lambda: sum(range(1000)))
print(elapsed >= 0)

True


### Expected output

```text
True
```

### Step-by-step explanation

Repeated timing reduces one-off noise, but realistic data, warm-up, correctness checks, startup, serialization, and resource limits still matter.

### What was learned

Benchmark end-to-end representative workloads, not tiny synthetic claims.

## Question 30 — Challenging

**Concept tested:** Resilient task runner

**Question:** Run thread tasks with per-task timeouts/errors and return successes plus failures.

**Hint:** Futures hold results and exceptions; the executor context waits during shutdown.

### Solution approach

Submit indexed jobs and classify each `result()`.

In [31]:
def maybe_fail(value):
    if value < 0: raise ValueError("negative")
    return value * 10
def run_tasks(values):
    successes, failures = {}, {}
    with ThreadPoolExecutor(max_workers=3) as pool:
        futures = {pool.submit(maybe_fail, value): value for value in values}
        for future, value in futures.items():
            try: successes[value] = future.result(timeout=1)
            except Exception as error: failures[value] = type(error).__name__
    return successes, failures
print(run_tasks([1, -1, 2]))

({1: 10, 2: 20}, {-1: 'ValueError'})


### Expected output

```text
({1: 10, 2: 20}, {-1: 'ValueError'})
```

### Step-by-step explanation

Each Future is observed independently, so one failure does not discard successful results.

### What was learned

Production concurrency needs bounded pools, observed failures, timeouts, cancellation policy, and clean shutdown.

# Assignment Revision Cheat Sheet

| Need | First choice |
|---|---|
| overlap I/O waits | threads / `ThreadPoolExecutor` |
| parallel pure-Python CPU work | processes / `ProcessPoolExecutor` |
| thread messages | `queue.Queue` |
| process messages | `multiprocessing.Queue` / `Pipe` |
| protect shared invariant | `Lock` / `RLock` |
| limit capacity | `Semaphore` |
| signal readiness/stop | `Event` |
| fixed phase checkpoint | `Barrier` |

- `start()` begins; `join()` waits; Future `result()` returns or re-raises.
- The GIL usually prevents CPU parallelism among ordinary CPython threads.
- Process workers must be top-level/picklable, and launch belongs under the main guard.
- Prefer message passing; synchronize deliberately when sharing memory.
- Bound worker counts/queues, design cancellation, handle errors, close resources, and benchmark representative workloads.